# Intro

In [3]:
ALPHA=0.05

## Modules

### Standard

In [4]:
import os, pickle, platform, sys, json
import numpy as np
import copy
import torch

In [25]:
import io
import tarfile

In [5]:
from collections import defaultdict

In [6]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [7]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [8]:
from scipy.stats import spearmanr

In [9]:
from tqdm.notebook import tqdm, trange

In [10]:
import datetime as dt

In [11]:
from bowtie import edges2bowtie

In [12]:
from tabulate import tabulate

In [13]:
from sklearn.metrics import jaccard_score as js
from sklearn.metrics import f1_score as f1
from sklearn.metrics import accuracy_score as accuracy
from sklearn.metrics import precision_score as precision
from sklearn.metrics import recall_score as recall
from sklearn.metrics import balanced_accuracy_score as b_accuracy

### Home made

In [14]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [15]:
from bowtie_plot_functions import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from bowtie_plot_functions import _fdr as fdr

In [16]:
from auxiliary_functions import el2ks
from auxiliary_functions import load_dico_edges as lde

Air!


## Folders

In [17]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER=HOME+'BiDCM/'
PLOT_FOLDER=HOME+'plots/'

In [18]:
if platform.system() == 'Darwin':
    os.system("afplay ta-da.mp3")
    PVALUE_FOLDER = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/benchmark_19/results_final/pvalues/'
elif platform.system() == 'Linux':
    
    PVALUE_FOLDER = '/home/sarawalk/bowtie2_py39/bowtie2/benchmark_19/results_final/pvalues/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")


## Names for $\LaTeX$
#### Names as in the draft

In [19]:
def parse_dico_names(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip().split() for line in f]
        # such a command creates a list
        # in which each element is a list of tokens of a line in the file
        # remarkably, there is an empty element
        # before each dataset name
        

    result = {}
    current_key = None

    for tokens in lines:
        if not tokens:
            current_key = None
        elif current_key is None:
            # prima riga non vuota del gruppo: nome del dataset
            current_key = tokens[0]
            result[current_key] = {}
        else:
            # riga tipo ['5:', 'journalists', '&', 'Media']
            idx = int(tokens[0].rstrip(':'))
            label = ' '.join(tokens[1:])
            result[current_key][idx] = label

    return result

In [20]:
cacca=parse_dico_names(GUARINO_FOLDER+'all_dico_labels.txt')

In [21]:
name_changer={'c0': 'Crisis - Centrists', 'c1': 'Crisis - Centre-right', 'c2': 'Crisis - M5S', 'c3': 'Crisis - PD', 'c4': 'Crisis - Media', 'e0': 'Elections - Centre-right', 'e1': 'Elections - Centre-left', 'e2': 'Elections - M5S', 'e3': 'Elections - Centrists', 'e4': 'Elections - Others', 'e5': 'Elections - Media', 'e6': 'Elections - Others 2', 'p0': 'President -  Centrists', 'p1': 'President -  PD', 'p2': 'President -  M5S', 'p3': 'President -  Media', 'p4': 'President -  Right', 'p5': 'President -  Others', 'p6': 'President -  FI'}

#### Proper scientific notation for the Latex table

In [22]:
def sci_latex(x, dec=2, bold=False):
    # da Claude
    mant, exp = f'{x:.{dec}e}'.split('e')
    if bold:
        return f'$\\mathbf{{{mant}\\times 10^{{{int(exp)}}}}}$'
    return f'${mant}\\times 10^{{{int(exp)}}}$'


## Functions

In [23]:
def p_value_right_tail(count_sample, obs, l_sample):
    """
    Compute the empirical p-value based on the count sample and observed value.

    Parameters:
    count_sample (array-like): The sample of counts from the null distribution.
    obs (float): The observed value to compare against the null distribution.
    l_sample (int): The number of samples in count_sample.

    Returns:
    float: The empirical p-value.
    """
    if len(count_sample) == 0:
        return 1.0
    else:
        return (np.sum(np.array(count_sample) >= obs) + 1) / (l_sample + 1)

In [24]:
def p_value_two_tailed(count_sample, obs, l_sample):
    """
    Compute the empirical two-tailed p-value based on the count sample and observed value.

    Parameters:
    count_sample (array-like): The sample of counts from the null distribution.
    obs (float): The observed value to compare against the null distribution.
    l_sample (int): The number of samples in count_sample.

    Returns:
    float: The empirical two-tailed p-value, 2*min(p_left, p_right), capped at 1.
    """
    if len(count_sample) == 0:
        return 1.0
    count_sample = np.asarray(count_sample)
    p_right = (np.sum(count_sample >= obs) + 1) / (l_sample + 1)
    p_left = (np.sum(count_sample <= obs) + 1) / (l_sample + 1)
    return min(1.0, 2 * min(p_left, p_right))


# Who is where?
Get who are the most active nodes per dico and per block, for each dataset.

## Degree and strengths

In [47]:
dataset_converter={'e':'i', 'p':'q'}
k_in_dict={}
k_out_dict={}
s_in_dict={}
s_out_dict={}
nodelist_dict={}

for key in tqdm(name_changer.keys()):
    # load the edgelist for the current dataset
    _converted_key = dataset_converter.get(key[0], key[0])
    _edgelist=lde(_converted_key, int(key[1]))
    # compute the in-degree, out-degree, in-strength, and out-strength for the current edgelist
    k_out, k_in, s_out, s_in, all_nodes=el2ks(_edgelist)
    k_in_dict[key]=k_in
    k_out_dict[key]=k_out
    s_in_dict[key]=s_in
    s_out_dict[key]=s_out
    nodelist_dict[key]=all_nodes

  0%|          | 0/19 [00:00<?, ?it/s]

## Bowtie

In [52]:
dataset_converter={'e':'i', 'p':'q'}
bowtie_dict={}

for key in tqdm(name_changer.keys()):
    # load the edgelist for the current dataset
    _converted_key = dataset_converter.get(key[0], key[0])
    _edgelist=lde(_converted_key, int(key[1]))
    # compute the in-degree, out-degree, in-strength, and out-strength for the current edgelist
    topo_el = [(s, t) for s, t, w in _edgelist]
    _aux = edges2bowtie(topo_el)   # node -> block label
    _inverted_aux=defaultdict(list)
    for node, block in _aux.items():
        _inverted_aux[block].append(node)
    bowtie_dict[key]=_inverted_aux
    

  0%|          | 0/19 [00:00<?, ?it/s]

## Names and roles

In [27]:
source='rawdata/'
os.listdir(source)

['crisi.KXEhUm2h.tar.gz.part',
 'crisi.tar.gz',
 'ita_election_campaign_2022.tar.gz',
 'quirinale2022.tar.gz']

In [28]:
# the order is the temporal one
raw_data=['quirinale2022.tar.gz','crisi.tar.gz','ita_election_campaign_2022.tar.gz']


In [29]:
user_dict = {}
with tarfile.open(source+'ita_election_campaign_2022.tar.gz', "r:gz") as tf:
    m = tf.next()  
    with io.TextIOWrapper(tf.extractfile(m), encoding="utf-8") as f:
        line = f.readline()
        while len(line) > 0:   
            _json_entry = json.loads(line)
            for user in _json_entry['includes']['users']:
                # if the entry is alaredy in the dictionary, it updates the existing entry with the new information
                if user['id'] not in user_dict.keys():
                    user_dict[user['id']] = {}
                    user_dict[user['id']]['username'] = user['username']
                    user_dict[user['id']]['name'] = user['name']
                    user_dict[user['id']]['description'] = user['description']
                    user_dict[user['id']]['verified'] = user['verified']
                    user_dict[user['id']]['followers_count']= user['public_metrics']['followers_count']
                    user_dict[user['id']]['following_count']= user['public_metrics']['following_count']
            print(f"Processed {len(user_dict):6,} users so far...", end="\r")
            line = f.readline()


In [30]:
os.system("afplay ta-da.mp3")

0

In [36]:
_json_entry['data'][0]['author_id']

'346720029'

In [40]:
_json_entry['data'][0].keys()

dict_keys(['created_at', 'reply_settings', 'author_id', 'edit_history_tweet_ids', 'conversation_id', 'referenced_tweets', 'entities', 'public_metrics', 'possibly_sensitive', 'id', 'source', 'text', 'context_annotations', 'lang'])

In [46]:
_json_entry['data'][0]['entities']['mentions']

[{'start': 3, 'end': 19, 'username': 'salvini_giacomo', 'id': '1451424235'}]

In [ ]:
n_check=0
with tarfile.open(source+'ita_election_campaign_2022.tar.gz', "r:gz") as tf:
    m = tf.next()  
    with io.TextIOWrapper(tf.extractfile(m), encoding="utf-8") as f:
        line = f.readline()
        while len(line) > 0:   
            _json_entry = json.loads(line)
            _data=_json_entry['data']
            for d in _data:
                assert d['author_id'] in user_dict.keys(), f"Author ID {d['author_id']} not found in user_dict"
                n_check+=1
                if 'referenced_tweets' in d and d['referenced_tweets'][0]['type'] == 'retweeted':
                    if 'entities' in d and 'mentions' in d['entities']:
                        # the first mention is the one that is retweeted, so we check that it is in the user_dict
                        mention=d['entities']['mentions'][0]
                        assert mention['id'] in user_dict.keys(), f"Mentioned username {mention['username']} not found in user_dict"
                        n_check+=1
            print(f"Processed {n_check:9,} users so far...", end="\r")
            line = f.readline()


In [55]:
with tarfile.open(source+'ita_election_campaign_2022.tar.gz', "r:gz") as tf:
    m = tf.next()  
    with io.TextIOWrapper(tf.extractfile(m), encoding="utf-8") as f:
        line = f.readline()


In [56]:
_json_entry =json.loads(line)

In [60]:
_json_entry['data'][0]['referenced_tweets'][0]['type']

'retweeted'